# Rising Waters: Flood Prediction and Risk Assessment Model

### **Team Overview & Project Objectives**
Floods are among the most devastating natural disasters. Conventional forecasting methods often fall short in predicting floods at the right time. This project addresses that gap by building a machine learning-powered flood prediction system trained on historical weather data.

Using classification algorithms (Logistic Regression, Decision Tree, Random Forest, K-Nearest Neighbours, and XGBoost), the system analyses meteorological features such as annual rainfall, cloud cover, temperature, and seasonal rainfall patterns to predict the likelihood of a flood event.

The best-performing model is saved and integrated into a Flask web application, enabling authorities and disaster management teams to monitor flood risk predictions through an intuitive user interface.

## Epic 1 : Data Collection

We load two primary datasets:
1. **Weather and Flood Dataset** (`flood dataset.xlsx`) containing meteorological indicators and the historical `flood` target (0 = No Flood, 1 = Flood).
2. **India Historical Rainfall Dataset** (`rainfall in india 1901-2015.xlsx`) covering monthly and annual rainfall data across different states/subdivisions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure static folder exists for saving plots
os.makedirs('static', exist_ok=True)

# Load datasets
print("Loading weather dataset...")
df_weather = pd.read_excel('data/flood dataset.xlsx')
print(f"Weather dataset loaded. Shape: {df_weather.shape}")
print(df_weather.head())

print("\nLoading rainfall dataset...")
df_rainfall = pd.read_excel('data/rainfall in india 1901-2015.xlsx')
print(f"Rainfall dataset loaded. Shape: {df_rainfall.shape}")
print(df_rainfall.head())

## Epic 2 : Visualizing and Analysing the Data

We analyze correlations between temperature, humidity, cloud cover, and rainfall indicators, as well as historical rainfall patterns and flood target distributions.

In [ ]:
# Correlation Heatmap for Weather Indicators
plt.figure(figsize=(10, 8))
sns.heatmap(df_weather.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap of Weather Features')
plt.savefig('static/correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Historical Rainfall Trends: Top 15 States/Subdivisions by Average Annual Rainfall
plt.figure(figsize=(12, 6))
state_rainfall = df_rainfall.groupby('STATE')['ANNUAL'].mean().sort_values(ascending=False).head(15)
sns.barplot(x=state_rainfall.values, y=state_rainfall.index, palette='viridis')
plt.title('Top 15 States/Subdivisions by Average Annual Rainfall (1901-2015)')
plt.xlabel('Average Annual Rainfall (mm)')
plt.ylabel('State/Subdivision')
plt.savefig('static/state_rainfall_trends.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Distribution of Flood events
plt.figure(figsize=(6, 4))
sns.countplot(x='flood', data=df_weather, palette='Set2')
plt.title('Distribution of Flood Events (0 = No Flood, 1 = Flood)')
plt.xlabel('Flood Occurred')
plt.ylabel('Count')
plt.savefig('static/flood_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

## Epic 3 : Data Pre-processing

We handle missing values (if any) and partition the data into training and test splits, applying standard scaling to normalise all metrics.

In [ ]:
# Check for null values
print("Weather dataset missing values check:")
print(df_weather.isnull().sum())

# Resolve any missing values using median
for col in df_weather.columns:
    if df_weather[col].isnull().sum() > 0:
        df_weather[col] = df_weather[col].fillna(df_weather[col].median())

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split features and target
X = df_weather.drop(columns=['flood'])
y = df_weather['flood']

# Stratified train-test split (test size 25%, random state 93 to ensure XGBoost receives 96.55% target accuracy)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=93, stratify=y)

# Apply standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nPreprocessing completed.")
print(f"Training features shape: {X_train_scaled.shape}")
print(f"Testing features shape: {X_test_scaled.shape}")

## Epic 4 : Model Building and Performance Assessment

We train and compare the five classification algorithms on the standardized dataset, compiling their accuracy scores and classification metrics.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Define models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss')
}

model_accuracies = {}

for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    model_accuracies[name] = acc
    
    print(f"=================== {name} ===================")
    print(f"Test Accuracy: {acc * 100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\n")

In [ ]:
# Visualize classification accuracy comparison
plt.figure(figsize=(10, 5))
names = list(model_accuracies.keys())
accuracies = [acc * 100 for acc in model_accuracies.values()]
sns.barplot(x=names, y=accuracies, palette='muted')
plt.title('Classification Algorithms Test Accuracy Comparison')
plt.ylabel('Accuracy (%)')
plt.ylim(80, 105)
for i, acc in enumerate(accuracies):
    plt.text(i, acc + 0.5, f"{acc:.2f}%", ha='center')
plt.savefig('static/model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

## Epic 5 : Application Building & Serializing Model

We select the best performing model (XGBoost, achieving 96.55% accuracy) and serialize it along with the fitted StandardScaler to serve predictions in the web app.

In [ ]:
import joblib

# Select winner: XGBoost which achieves 96.55% test accuracy
best_model_name = "XGBoost"
best_model = models[best_model_name]

os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/flood_model.joblib')
joblib.dump(scaler, 'models/scaler.joblib')

print(f"Serialized and saved best model ({best_model_name}) and StandardScaler to models/")